# Embedding Process - Multi-Model Generation

Questo notebook documenta il processo completo che porta dal dataset preprocessato agli embeddings numerici per **tre modelli di embedding** utilizzati nelle fasi di clustering.

La logica rimane nei moduli `src/utils`; qui vengono spiegati i passaggi, generati gli embeddings per ogni modello, caricati gli output e verificata la coerenza.

## Obiettivo

Generare embeddings per tre modelli diversi su uno stesso dataset processed:
- `BAAI/bge-small-en-v1.5` (384 dim, veloce e leggero)
- `sentence-transformers/all-MiniLM-L6-v2` (384 dim, ultra-veloce)
- `intfloat/e5-base-v2` (768 dim, alta qualità)

## 1. Setup

Carichiamo path, configurazione e funzioni di supporto. Gli embeddings generati per ogni modello verranno salvati in `data/embeddings` con suffisso model-specifico.

In [ ]:
from pathlib import Path
import json
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.embedding_pipeline import (
    EmbeddingConfig, 
    MODEL_REGISTRY, 
    build_embedding_jobs,
    load_embedding_model,
    encode_texts,
    aggregate_chunk_embeddings,
)

ENV_PATH = PROJECT_ROOT / ".env"
ENV = dotenv_values(ENV_PATH)

def project_path(env_key, default):
    value = Path(ENV.get(env_key, default))
    return value if value.is_absolute() else PROJECT_ROOT / value

def get_model_slug(model_name: str) -> str:
    """Generate a short slug from model full name for filenames."""
    slug = model_name.split('/')[-1]
    slug = slug.replace('.', '-')
    return slug

BASE_CONFIG = EmbeddingConfig.from_env(ENV_PATH)
PROCESSED_PATH = BASE_CONFIG.processed_input_path

EMBEDDING_MODELS = [
    "BAAI/bge-small-en-v1.5",
    "sentence-transformers/all-MiniLM-L6-v2",
    "intfloat/e5-base-v2",
]

EMBEDDINGS_ROOT = project_path("DATA_EMBEDDINGS_PATH", "data/embeddings/")
METADATA_ROOT = project_path("METADATA_PATH", "data/metadata/")
FIGURES_ROOT = project_path("FIGURES_PATH", "reports/figures/")
FIGURES_PATH = FIGURES_ROOT / ENV.get("EMBEDDING_NOTEBOOK_FIGURES_SUBDIR", "embedding_process")
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

plt.style.use("default")
pd.set_option("display.max_columns", 80)

print(f"Modelli da generare: {len(EMBEDDING_MODELS)}")
for i, model in enumerate(EMBEDDING_MODELS, 1):
    print(f"  {i}. {model}")

## 2. Caricamento dataset processed

Leggiamo il dataset preprocessato che sarà usato per tutti i modelli.

In [ ]:
processed = pd.read_parquet(PROCESSED_PATH)
print(f"Righe: {len(processed):,}")
print(f"Colonne: {processed.shape[1]}")
print("\nSample dati:")
processed[["id", "combined_text_length", "has_redaction"]].head()

## 3. Generazione embeddings per ogni modello

Iteriamo su ogni modello e generiamo gli embeddings.

In [ ]:
results = []
all_model_data = {}

for model_name in EMBEDDING_MODELS:
    slug = get_model_slug(model_name)
    print(f"\nModello: {model_name}")
    
    embeddings_path = EMBEDDINGS_ROOT / f"email_embeddings_{slug}.npy"
    index_path = METADATA_ROOT / f"email_embedding_index_{slug}.parquet"
    metadata_path = METADATA_ROOT / f"email_embedding_metadata_{slug}.json"
    
    artifacts_exist = embeddings_path.exists() and index_path.exists() and metadata_path.exists()
    
    if artifacts_exist:
        print(f"  ✓ Caricando artefatti esistenti...")
        embeddings = np.load(embeddings_path)
        embedding_index = pd.read_parquet(index_path)
        with metadata_path.open(encoding="utf-8") as f:
            embedding_metadata = json.load(f)
        status = "reused"
        gen_time = None
    else:
        print(f"  ⚙ Generando embeddings...")
        start = time.perf_counter()
        
        config = EmbeddingConfig(
            model_name=model_name,
            input_text_column="combined_text",
            embedding_text_column="embedding_text",
            chunk_char_length=1800,
            chunk_char_overlap=200,
            processed_input_path=PROCESSED_PATH,
            embeddings_output_path=embeddings_path,
            embedding_index_output_path=index_path,
            embedding_metadata_output_path=metadata_path,
            batch_size=32,
        )
        
        chunk_texts, embedding_index = build_embedding_jobs(processed, config)
        model = load_embedding_model(model_name)
        chunk_embeddings = encode_texts(model, chunk_texts, batch_size=32)
        embeddings = aggregate_chunk_embeddings(chunk_embeddings, embedding_index)
        
        embeddings_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(embeddings_path, embeddings)
        
        index_path.parent.mkdir(parents=True, exist_ok=True)
        embedding_index.to_parquet(index_path, index=False)
        
        embedding_metadata = {
            "model_name": model_name,
            "shape": list(embeddings.shape),
            "chunk_count": len(chunk_texts),
        }
        metadata_path.parent.mkdir(parents=True, exist_ok=True)
        metadata_path.write_text(json.dumps(embedding_metadata, indent=2, ensure_ascii=False), encoding="utf-8")
        
        gen_time = time.perf_counter() - start
        status = "generated"
        print(f"  ✓ Completato in {gen_time:.2f}s")
    
    norms = np.linalg.norm(embeddings, axis=1) if embeddings.size else np.array([])
    
    results.append({
        "model": model_name,
        "slug": slug,
        "status": status,
        "shape": embeddings.shape,
        "norm_mean": float(norms.mean()) if norms.size else 0.0,
    })
    
    all_model_data[model_name] = {
        "embeddings": embeddings,
        "index": embedding_index,
        "norms": norms,
    }

print("\n" + "="*60)
print("RIEPILOGO GENERAZIONE")
print("="*60)
for r in results:
    print(f"{r['slug']:20} | shape: {r['shape']} | norm_mean: {r['norm_mean']:.4f}")

## 4. Grafici diagnostici

Distribuzione norme e chunk count per ogni modello.

In [ ]:
fig, axes = plt.subplots(len(EMBEDDING_MODELS), 2, figsize=(14, 4 * len(EMBEDDING_MODELS)))
if len(EMBEDDING_MODELS) == 1:
    axes = np.array([axes])

for row_idx, model_name in enumerate(EMBEDDING_MODELS):
    slug = get_model_slug(model_name)
    data = all_model_data[model_name]
    norms = data["norms"]
    chunk_counts = data["index"]["chunk_count"]
    
    axes[row_idx, 0].hist(norms, bins=30)
    axes[row_idx, 0].set_title(f"{slug}: Norme L2")
    axes[row_idx, 0].set_xlabel("Norma")
    axes[row_idx, 0].set_ylabel("Frequenza")
    
    chunk_dist = chunk_counts.value_counts().sort_index()
    axes[row_idx, 1].bar(chunk_dist.index.astype(str), chunk_dist.values)
    axes[row_idx, 1].set_title(f"{slug}: Chunk per email")
    axes[row_idx, 1].set_xlabel("Chunk count")
    axes[row_idx, 1].set_ylabel("Email")

fig.tight_layout()
fig.savefig(FIGURES_PATH / "embedding_single_model_diagnostics.png", dpi=150)
plt.show()
print(f"Figura salvata: {FIGURES_PATH / 'embedding_single_model_diagnostics.png'}")

## 5. Conclusione

La pipeline ha generato embeddings per tre modelli. Prossimi passi: valutazione clustering e metriche.